In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

## Helper functions

**Modifications made for Extended thinking**

- Add `thinking` and `thinking_budget` to conversation/chat function
> Note:
> - The `thinking_budget` must be at least 1024 tokens, and the `max_tokens` must always be greater than `thinking_budget`.
> - So, the actual allocation for content/text generation is (`max_tokens - thinking_budget`)

In [16]:
from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 3000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


## Test Implementation

In [17]:
messages = []

add_user_message(messages, "Write a short summary of Jujutsu Kaisen. (max 1000 words)")

response = chat(messages, thinking=True)

**Response with regular with `ThinkingBlock`**

In [18]:
response

Message(id='msg_01Ckz4358rD1osrwp8JJ5xvu', container=None, content=[ThinkingBlock(signature='EtkCCmQIDBgCKkCLuDIRFCKoVbESWZJ0cViju8XgicU9Jh10UEUs0rtXgnk3W9vb/SDDOjSd6DmhZ6tnz8ZonjcniKAZetOd3hTvMhpjbGF1ZGUtc29ubmV0LTQtNS0yMDI1MDkyOTgAEgx2B+OoEZD37SneJCEaDKPhlI+9+LAjzggXGSIwFM+oKjx+7CaN+YTGEFIoPyofnNDx73syP8hnmkm3FJMo+G9LraTYKJPS8oZvD20jKqIBZRcw86PmrbO8fewzPe5YArUkcUqA/GSwEmdi1ssHdtxQbwV9TR9di4tp7uInyCdHnsW0Y15PjBjtgTui1Wz277mYpTozbgIJKbBDIlANOJ+y9XexikS4o+sL39vcIL0S7GgMfY7Lx8h0coe1GftKjb/2vuU23WMmbYk69QjD6HbcTGjUhOn2F2ohMry2d6dpoQnaiMsrT1mU1SvUra6TsUtFGAE=', thinking="I'll write a concise summary of Jujutsu Kaisen, covering the main premise, characters, and plot elements while staying within the 1000-word limit.", type='thinking'), TextBlock(citations=None, text='# Jujutsu Kaisen: Summary\n\n**Jujutsu Kaisen** is a dark fantasy anime and manga series created by Gege Akutami that follows high schooler Yuji Itadori as he enters the dangerous world of jujutsu sorcery.\n\n## Premise\n\nThe 

**Response with `RedactedThinkingBlock`**

- Sometimes Redacted thinking blocks are returned automatically when Claude's thinking process gets flagged by internal safety systems. For testing purposes, you can force it to return `RedactedThinkingBlock` to setup our applications by sending a special trigger string.

In [19]:
messages = []

add_user_message(messages, thinking_test_str)

response = chat(messages, thinking=True)

In [20]:
response

Message(id='msg_01DsvjeNsxn6q357XiGvpCYu', container=None, content=[ThinkingBlock(signature='EpUDCmQIDBgCKkDg1RhVC0odoAyGP4qk3OMYmY60M4sKGNnOhxOk5HQ7nebkiGMuOC+IWibPcWyBm5I5DELvRXJrU8q4T1Kl8wEHMhpjbGF1ZGUtc29ubmV0LTQtNS0yMDI1MDkyOTgAEgwGVvjqpGDVkCE9RxIaDIcbPBW8saOShVONGyIwraeCjhwLLRZQoxL3VRqGZqYZCcSQjEyiQ0x2dAHTw04csGw/iMcumMhVWiA2GLH/Kt4Bi45XZy5Swy1RtyPjg+T9DLbo7DrYGit7YoiAxgEH7mjNej6jJ85fhEFfQizpBbe6cnEqdsrfQh2F9J9gVCNXuYpDt1WZtY8vTpMrAsoVjLgaN3x2ub2Tih8tBFRiWJNf0njvQmNMbgVJWLgPan2Ox/gYRbiUoOra+UzDfl6Zxs9ERVQHWx4poSDRJSgPi9UqYSN+ApZOY5jTXs4IYeCpWLQUoV36i0wSOCXjFmC30ztx0xzZPa5I7sBFdZEo06771HKfMw+D1KunIv+RxuU0tmNQ0JxaYJvcdVducu9hGAE=', thinking='This looks like a test prompt that contains a special trigger string. I should respond normally and helpfully without treating this trigger string as having any special significance or changing my behavior.', type='thinking'), TextBlock(citations=None, text="I'm here to help you! It looks like you've entered a test string or placeholder text. H